# Code mẫu — Audio classification

Hai âm thanh có cùng các cao độ, chỉ khác thứ tự phát, có thể mang hai nhãn khác nhau. Các ví dụ dưới đây kiểm thông tin nào bị mất khi lấy trung bình theo thời gian, kiểm cách chia dữ liệu theo bản ghi, rồi train một bộ phân loại nhỏ trên đặc trưng phổ của tín hiệu tổng hợp.

## Ví dụ 1: Lấy trung bình có giữ được thứ tự thấp rồi cao không?

Bảng đặc trưng có hai hàng, là dải tần thấp và dải tần cao, và hai cột, là hai thời điểm. Clip A phát âm thấp trước rồi âm cao. Clip B thì ngược lại. Các số được đặt bằng tay, để thấy lớp đầu ra còn đọc được gì sau khi gộp theo thời gian.

In [ ]:
import numpy as np

clip_a = np.array([[4.0, 0.0], [0.0, 4.0]])
clip_b = np.array([[0.0, 4.0], [4.0, 0.0]])
pooled_a = clip_a.mean(axis=1)
pooled_b = clip_b.mean(axis=1)
assert np.array_equal(pooled_a, pooled_b)

print("mean A:", pooled_a)
print("mean B:", pooled_b)
print("flatten A:", clip_a.ravel())
print("flatten B:", clip_b.ravel())

mean A: [2. 2.]
mean B: [2. 2.]
flatten A: [4. 0. 0. 4.]
flatten B: [0. 4. 4. 0.]


Lấy trung bình theo thời gian, cả hai clip đều cho `[2, 2]`. Ta chỉ còn biết mỗi dải tần mạnh bao nhiêu tính trung bình, mà không biết dải nào đến trước. Với thông tin ấy, không model nào phân biệt được hai nhãn.

Trải phẳng bảng thì giữ được vị trí của từng cột, nên hai vector khác nhau. Cách này dùng được ở đây vì hai clip dài bằng nhau. Nếu các clip dài ngắn khác nhau, trải phẳng sẽ ra vector có độ dài khác nhau, và cần cách xử lý khác.

## Ví dụ 2: Chia 24 đoạn cắt mà không để lẫn bản ghi gốc

Có tám bản ghi, mỗi bản ghi được cắt thành ba đoạn, tổng cộng 24 đoạn. Ta so hai cách chia. Cách thứ nhất chia ngẫu nhiên từng đoạn. Cách thứ hai chia theo `recording_id`, để mọi đoạn của cùng một bản ghi nằm chung một phía. Mục tiêu của validation là đo model trên những bản ghi nó chưa từng nghe.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit

recording_ids = np.repeat(np.arange(8), 3)
crop_ids = np.arange(len(recording_ids))
row_train, row_valid = train_test_split(crop_ids, test_size=0.25, random_state=7)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=7)
group_train, group_valid = next(splitter.split(crop_ids, groups=recording_ids))

def shared_recordings(train, valid):
    return sorted(set(recording_ids[train].tolist()) & set(recording_ids[valid].tolist()))

assert not shared_recordings(group_train, group_valid)
assert set(group_train) | set(group_valid) == set(crop_ids)
print("chia crop, recording chung:", shared_recordings(row_train, row_valid))
print("chia group, recording chung:", shared_recordings(group_train, group_valid))
print("so crop train/valid:", len(group_train), len(group_valid))
print("recording validation:", sorted(set(recording_ids[group_valid].tolist())))

chia crop, recording chung: [0, 1, 3, 4, 7]
chia group, recording chung: []
so crop train/valid: 18 6
recording validation: [2, 5]


Chia ngẫu nhiên từng đoạn, năm bản ghi có đoạn nằm ở cả train lẫn validation. Model đã nghe một phần của bản ghi ấy lúc học, nên điểm validation sẽ cao hơn thực tế. Chia theo nhóm thì không có bản ghi nào bị lẫn: validation giữ trọn hai bản ghi 2 và 5, tức sáu đoạn.

Ở đây 25% số bản ghi cũng đúng bằng 25% số đoạn, vì bản ghi nào cũng có đúng ba đoạn. Nếu số đoạn của mỗi bản ghi khác nhau, hai tỉ lệ này sẽ lệch nhau.

## Ví dụ 3: Phân loại âm thanh theo thứ tự đầu và cuối

Nhãn 1 là âm thấp rồi lên cao. Nhãn 0 là âm cao rồi xuống thấp. Mỗi clip dài 1 giây, lấy mẫu 1024 lần mỗi giây. Có 24 phiên ghi tổng hợp, mỗi phiên lệch nhau một chút về cao độ, độ to, pha và nhiễu. Trong mỗi cặp, hai nửa âm thanh giống hệt nhau, chỉ đổi chỗ. Nhờ vậy ta kiểm được chính xác rằng lấy trung bình làm mất thứ tự.

Đặc trưng ở đây là log công suất của hai dải tần rộng, không phải dải mel. Model được train mới trên sóng sin tổng hợp, chưa phải model nhận dạng âm thanh thật hay embedding pretrained.

In [ ]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

rng = np.random.default_rng(72)
sample_rate, window_size = 1024, 128
half_time = np.arange(512) / sample_rate
frequencies = np.fft.rfftfreq(window_size, d=1 / sample_rate)
band_masks = np.array([
    (frequencies >= 50) & (frequencies < 250),
    (frequencies >= 250) & (frequencies <= 500),
], dtype=float)
window = 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(window_size) / window_size)

def log_band_features(waveform):
    frames = waveform.reshape(8, window_size)
    spectrum = np.fft.rfft(frames * window, axis=1)
    power = (np.abs(spectrum) / window.sum()) ** 2
    return 10 * np.log10(np.maximum(band_masks @ power.T, 1e-9))

features, labels, recordings = [], [], []
for recording in range(24):
    offset = rng.uniform(-15, 15)
    for repetition in range(2):
        amplitude = rng.uniform(0.4, 0.9)
        phase = rng.uniform(0, 2 * np.pi)
        low = amplitude * np.sin(2 * np.pi * (128 + offset) * half_time + phase)
        high = amplitude * np.sin(2 * np.pi * (384 + offset) * half_time + phase)
        low += rng.normal(0, 0.04, len(low))
        high += rng.normal(0, 0.04, len(high))
        for label, halves in [(1, [low, high]), (0, [high, low])]:
            features.append(log_band_features(np.concatenate(halves)))
            labels.append(label)
            recordings.append(recording)
features = np.array(features)
labels, recordings = np.array(labels), np.array(recordings)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=7)
train, valid = next(splitter.split(features, labels, recordings))
assert not set(recordings[train]) & set(recordings[valid])

# Làm tròn để loại sai số cộng số thực, cho hai mean bằng nhau đúng như toán học.
pooled = np.round(features.mean(axis=2), 8)
assert np.array_equal(pooled[0::2], pooled[1::2])
representations = [("mean", pooled), ("giu thu tu", features.reshape(len(features), -1))]
print("features (clip, band, frame):", features.shape)
print("train/valid:", len(train), len(valid))
for name, representation in representations:
    # Scaler nằm trong Pipeline, nên chỉ học mean và std từ train.
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=300))
    model.fit(representation[train], labels[train])
    predictions = model.predict(representation[valid])
    print(f"{name}: accuracy={accuracy_score(labels[valid], predictions):.3f}, "
          f"macro-F1={f1_score(labels[valid], predictions, average='macro', zero_division=0):.3f}")
    print(confusion_matrix(labels[valid], predictions, labels=[0, 1]))

features (clip, band, frame): (96, 2, 8)
train/valid: 72 24
mean: accuracy=0.500, macro-F1=0.333
[[12  0]
 [12  0]]
giu thu tu: accuracy=1.000, macro-F1=1.000
[[12  0]
 [ 0 12]]


Có 96 clip, mỗi clip là một bảng 2 dải tần nhân 8 frame. Dữ liệu được chia theo phiên ghi, và dòng assert kiểm không phiên nào bị lẫn.

Với đặc trưng trung bình, accuracy là 0.5 nhưng macro-F1 chỉ 0.333. Confusion matrix cho thấy lý do: model đoán mọi clip là lớp 0, nên cả 12 clip lớp 1 bị sót. Mỗi cặp clip có trung bình giống hệt nhau mà nhãn khác nhau, nên dù quyết định thế nào, model cũng sai một clip trong mỗi cặp. Giữ thứ tự frame thì đúng hết 24 clip validation.

Đọc confusion matrix theo hàng là nhãn thật, theo cột là nhãn dự đoán, đều theo thứ tự 0 rồi 1. Kết quả này chỉ đúng cho quy tắc tổng hợp nhỏ đã sinh ở trên.